## Setting Up....

In [1]:
home_dir = normalizePath("~")
relative_path = file.path("R programs")
full_path = file.path(home_dir, relative_path)
setwd(full_path)
getwd()

[1] "C:/Users/death-star/Documents/R programs"

In [2]:
library(readxl)
library(caret)
library(lubridate)
library(moments)
library(car)
library(lmtest)
library(sandwich)
library(tidyverse)
library(margins)

Loading required package: ggplot2

Loading required package: lattice


Attaching package: 'lubridate'


The following objects are masked from 'package:base':

    date, intersect, setdiff, union


Loading required package: carData

Loading required package: zoo


Attaching package: 'zoo'


The following objects are masked from 'package:base':

    as.Date, as.Date.numeric


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr   1.1.4     ✔ stringr 1.5.1
✔ forcats 1.0.0     ✔ tibble  3.2.1
✔ purrr   1.0.2     ✔ tidyr   1.3.1
✔ readr   2.1.5     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
✖ purrr::lift()   masks caret::lift()
✖ dplyr::recode() masks car::recode()
✖ purrr::some()   masks car::some()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [3]:
Data <- read_excel("week_6/ABC.xlsx")
class(Data$Date)
Data$Date = as.Date(Data$Date)
class(Data$Date)

[1] "POSIXct" "POSIXt"

[1] "Date"

### Create updown 1 returns are positive and 0 when returns are negative

In [4]:
Data = Data |> mutate(updown= ifelse(ABC>0, 1, 0))
Data = na.omit(Data)
head(Data)

Date,Price,ABC,Sensex,DividendAnnounced,Sentiment,Nifty,updown
<date>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2000-01-03,718.15,0.07992481,0.073772129,0,0.04893645,0.095816410,1
2000-01-04,712.90,-0.00731045,0.021562349,0,-0.05503706,0.009706008,0
2000-01-05,730.00,0.02398653,-0.024405346,0,0.01913459,-0.032213609,1
2000-01-06,788.35,0.07993151,0.012045921,0,0.08035507,0.011204936,1
2000-01-07,851.40,0.07997717,-0.001300371,0,0.09403754,-0.000397248,1
2000-01-10,919.50,0.07998591,0.019191132,1,0.01522908,0.030167565,1


## Splittin data into trainging and testing dataset

In [5]:
Data = Data |>filter(year(Date)>2006)
indx = sample(1:nrow(Data), as.integer(0.8*nrow(Data)))
train = Data[indx,]
test = Data[-indx,]
head(train)
head(test)

Date,Price,ABC,Sensex,DividendAnnounced,Sentiment,Nifty,updown
<date>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2013-09-24,345.50,0.010529395,0.029099035,0,-0.006661443,0.025940841,1
2018-04-02,277.90,0.002887044,-0.004381682,0,-0.064521519,-0.018197841,1
2015-05-11,318.60,0.019520000,-0.000931718,0,-0.053358369,0.006982114,1
2016-10-14,285.15,-0.009035621,-0.003021627,0,-0.027809165,-0.002718976,0
2008-04-29,188.60,0.027233115,0.036743843,0,0.024574895,0.027888257,1
2013-04-10,314.45,-0.016575450,-0.004314682,0,-0.090436013,0.001202628,0


Date,Price,ABC,Sensex,DividendAnnounced,Sentiment,Nifty,updown
<date>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2007-01-03,156.10,-0.016073117,-0.006464859,0,0.004034366,-0.010663054,0
2007-02-07,166.15,-0.001202284,-0.010105645,0,0.030947328,0.009739389,0
2007-02-08,166.70,0.003310262,0.008463853,0,-0.022284113,0.005941379,1
2007-02-22,150.70,-0.028055466,-0.008626001,0,-0.071487123,-0.028628417,0
2007-02-23,153.90,0.021234240,0.001393648,0,-0.076283973,0.010304051,1
2007-02-26,153.75,-0.000974659,0.003375603,0,0.084578405,-0.009569511,0


In [6]:
train = train |>arrange(Date)
test = test|>arrange(Date)

In [7]:
prop.table(table(Data$updown))*100
prop.table(table(train$updown))*100
prop.table(table(test$updown))*100


       0        1 
49.63942 50.36058 


       0        1 
50.03757 49.96243 


       0        1 
48.04805 51.95195 

In [8]:
linear = lm(updown~ Sensex, data = train)
summary(linear)


Call:
lm(formula = updown ~ Sensex, data = train)

Residuals:
    Min      1Q  Median      3Q     Max 
-1.6677 -0.4615 -0.0017  0.4500  1.3316 

Coefficients:
             Estimate Std. Error t value Pr(>|t|)    
(Intercept)  0.496237   0.009049   54.84   <2e-16 ***
Sensex      12.523484   0.630560   19.86   <2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 0.4668 on 2660 degrees of freedom
Multiple R-squared:  0.1291,	Adjusted R-squared:  0.1288 
F-statistic: 394.5 on 1 and 2660 DF,  p-value: < 2.2e-16


In [9]:
fitted.results = ifelse(linear$fitted.values>0.4, 1 , 0)
CM = confusionMatrix(as.factor(fitted.results), as.factor(train$updown))
performance_4 = tibble(Threshold =0.4, Accuracy = CM$overall["Accuracy"],Sensitivity = CM$byClass["Sensitivity"],
                Specificity = CM$byClass["Specificity"])

In [10]:
fitted.results = ifelse(linear$fitted.values>0.6, 1 , 0)
CM = confusionMatrix(as.factor(fitted.results), as.factor(train$updown))
performance_6 = tibble(Threshold =0.6, Accuracy = CM$overall["Accuracy"],Sensitivity = CM$byClass["Sensitivity"],
                Specificity = CM$byClass["Specificity"])

In [11]:
fitted.results = ifelse(linear$fitted.values>0.8, 1 , 0)
CM = confusionMatrix(as.factor(fitted.results), as.factor(train$updown))
performance_8 = tibble(Threshold =0.8, Accuracy = CM$overall["Accuracy"],Sensitivity = CM$byClass["Sensitivity"],
                Specificity = CM$byClass["Specificity"])

In [12]:
LinearPerformance = rbind(performance_4,performance_6, performance_8)
LinearPerformance$Class = "Linear"
LinearPerformance

Threshold,Accuracy,Sensitivity,Specificity,Class
<dbl>,<dbl>,<dbl>,<dbl>,<chr>
0.4,0.6063110,0.3070571,0.90601504,Linear
0.6,0.6160781,0.9136637,0.31804511,Linear
0.8,0.5255447,0.9932432,0.05714286,Linear


### Logit performance object

In [13]:
logit = glm(formula(linear), data = train, family = binomial('logit'))
null =glm(updown~1, data = train, family = binomial('logit'))
PseudoRsq = 1-logLik(logit)/logLik(null)
margins(logit)

,Date,Price,ABC,Sensex,DividendAnnounced,Sentiment,Nifty,updown,fitted,se.fitted,dydx_Sensex,Var_dydx_Sensex,_weights,_at_number
,<date>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<mrgnlffc>,<dbl>,<dbl>,<int>
1,2007-01-01,153.20,-0.018577835,0.020609571,0,-0.054241830,0.002635723,0,0.8305250,0.01418912,11.059103,0.5625194,NA,1
2,2007-01-02,158.65,0.035574413,0.023273919,0,0.008467111,0.010307882,1,0.8579893,0.01363778,9.573473,0.5625194,NA,1
3,2007-01-04,157.05,0.006085842,0.004787963,0,0.097575403,0.020007297,1,0.5857091,0.01126542,19.064332,0.5625194,NA,1
4,2007-01-05,159.00,0.012416428,-0.003839180,0,-0.056319409,0.003910649,1,0.4178571,0.01126794,19.111355,0.5625194,NA,1
5,2007-01-08,160.60,0.010062893,0.020390317,0,0.079457951,0.014303838,1,0.8280865,0.01422221,11.185281,0.5625194,NA,1
6,2007-01-09,156.40,-0.026151930,0.002224875,0,-0.068338808,-0.009343922,0,0.5361553,0.01064301,19.538706,0.5625194,NA,1
7,2007-01-10,156.45,0.000319693,0.014961593,0,0.027008654,0.028965824,1,0.7587051,0.01431855,14.383726,0.5625194,NA,1
8,2007-01-11,161.75,0.033876638,0.005733296,0,-0.006122434,0.006707490,1,0.6036099,0.01158230,18.798114,0.5625194,NA,1
9,2007-01-12,161.40,-0.002163833,0.000781416,0,0.058066754,-0.012296953,0,0.5078649,0.01051087,19.636533,0.5625194,NA,1


### Probit Model

In [14]:
fitted.results = ifelse(linear$fitted.values>0.4, 1 , 0)
CM = confusionMatrix(as.factor(fitted.results), as.factor(train$updown))
performance_4 = tibble(Threshold =0.4, Accuracy = CM$overall["Accuracy"],Sensitivity = CM$byClass["Sensitivity"],
                Specificity = CM$byClass["Specificity"])

In [15]:
fitted.results = ifelse(linear$fitted.values>0.6, 1 , 0)
CM = confusionMatrix(as.factor(fitted.results), as.factor(train$updown))
performance_6 = tibble(Threshold =0.6, Accuracy = CM$overall["Accuracy"],Sensitivity = CM$byClass["Sensitivity"],
                Specificity = CM$byClass["Specificity"])

In [16]:
fitted.results = ifelse(linear$fitted.values>0.8, 1 , 0)
CM = confusionMatrix(as.factor(fitted.results), as.factor(train$updown))
performance_6 = tibble(Threshold =0.8, Accuracy = CM$overall["Accuracy"],Sensitivity = CM$byClass["Sensitivity"],
                Specificity = CM$byClass["Specificity"])

In [17]:
LinearPerformance = rbind(performance_4,performance_6, performance_8)
LinearPerformance$Class = "Linear"
LinearPerformance

Threshold,Accuracy,Sensitivity,Specificity,Class
<dbl>,<dbl>,<dbl>,<dbl>,<chr>
0.4,0.6063110,0.3070571,0.90601504,Linear
0.8,0.5255447,0.9932432,0.05714286,Linear
0.8,0.5255447,0.9932432,0.05714286,Linear


In [18]:
probit = glm(formula(linear), data = train, family = binomial('probit'))
null =glm(updown~1, data = train, family = binomial('probit'))
PseudoRsq = 1-logLik(probit)/logLik(null)
margins(probit)

,Date,Price,ABC,Sensex,DividendAnnounced,Sentiment,Nifty,updown,fitted,se.fitted,dydx_Sensex,Var_dydx_Sensex,_weights,_at_number
,<date>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<mrgnlffc>,<dbl>,<dbl>,<int>
1,2007-01-01,153.20,-0.018577835,0.020609571,0,-0.054241830,0.002635723,0,0.8157527,0.01513663,11.79226,0.5232626,NA,1
2,2007-01-02,158.65,0.035574413,0.023273919,0,0.008467111,0.010307882,1,0.8454947,0.01495890,10.53147,0.5232626,NA,1
3,2007-01-04,157.05,0.006085842,0.004787963,0,0.097575403,0.020007297,1,0.5786966,0.01087254,17.32364,0.5232626,NA,1
4,2007-01-05,159.00,0.012416428,-0.003839180,0,-0.056319409,0.003910649,1,0.4271893,0.01085274,17.37344,0.5232626,NA,1
5,2007-01-08,160.60,0.010062893,0.020390317,0,0.079457951,0.014303838,1,0.8131559,0.01513650,11.89512,0.5232626,NA,1
6,2007-01-09,156.40,-0.026151930,0.002224875,0,-0.068338808,-0.009343922,0,0.5338858,0.01030586,17.60474,0.5232626,NA,1
7,2007-01-10,156.45,0.000319693,0.014961593,0,0.027008654,0.028965824,1,0.7418789,0.01441801,14.31204,0.5232626,NA,1
8,2007-01-11,161.75,0.033876638,0.005733296,0,-0.006122434,0.006707490,1,0.5950011,0.01117023,17.16518,0.5232626,NA,1
9,2007-01-12,161.40,-0.002163833,0.000781416,0,0.058066754,-0.012296953,0,0.5084214,0.01018598,17.66458,0.5232626,NA,1


In [19]:
cor(logit$fitted.values, probit$fitted.values)

[1] 0.9993087

In [20]:
Corr_ob = cbind.data.frame(linear$fitted.values, logit$fitted.values, probit$fitted.values)

In [21]:
cor(Corr_ob)

,linear$fitted.values,logit$fitted.values,probit$fitted.values
linear$fitted.values,1.0000000,0.9300510,0.9385632
logit$fitted.values,0.9300510,1.0000000,0.9993087
probit$fitted.values,0.9385632,0.9993087,1.0000000


### ROC curve


In [ ]:
pr = prediction(linear$fitted.values,train$updown)
prf = performan

Not done